# QR Phishing Detection — URL Model Training (Colab)

Trains a character-level CNN to detect phishing **URLs**. A QR code is just a container for a
URL, so the phishing signal is in the URL text. The app scans the QR image, decodes it to a URL,
and this model judges the URL.

**Before running:** download `malicious_phish.csv` from
https://www.kaggle.com/datasets/sid321axn/malicious-urls-dataset and put it in your Google Drive
at `/content/drive/MyDrive/malicious_phish.csv`. Then: **Runtime → Run all**.

## How it works — phishing detection logic & training

This notebook builds the machine-learning model that decides whether a QR code's link
is **safe** or **phishing**. Below is exactly how it detects phishing and how it is trained.

### 1. The problem ("Quishing")
Attackers hide a **phishing URL** inside a QR code. People scan it without thinking and
land on a fake login / payment page. We must judge the link **before** the user opens it.

### 2. Why we analyze the URL (not the QR picture)
A QR code is just a **lossless encoding of text** — a safe QR and a phishing QR look like
the same black-and-white squares. The danger is in the **URL inside**. So the pipeline is:

```
QR image  ──decode──▶  URL text  ──ML model──▶  P(phishing)  ──▶  safe / risky / dangerous
```
This notebook trains the **ML model** step.

### 3. How the model "checks" if a URL is phishing (the core logic)
This is **supervised machine learning** — the model is NOT given hand-written rules.
Instead it is shown thousands of example URLs that are already labelled:

| Example URL | Label |
|---|---|
| `https://github.com/login` | 0 = safe |
| `https://meta.com` | 0 = safe |
| `http://paypal-verify.tk/login` | 1 = phishing |
| `http://192.168.0.1@secure.cf/update` | 1 = phishing |

From these examples the model **learns the statistical patterns by itself** that separate
phishing from safe — typically things like:
- suspicious top-level domains: `.tk`, `.cf`, `.ga`, `.xyz`
- sensitive words in the path: `login`, `verify`, `secure`, `update`, `account`, `password`
- raw IP addresses, the `@` credential trick, many sub-domains, hyphens/digits, look-alike
  brand names (`paypa1`, `amaz0n`)
- versus the clean shape of real, well-known domains (which is why we also feed it tens of
  thousands of **real legit domains** in Section 2)

After training, for any **new** URL it outputs a number from **0.0 (safe) to 1.0 (phishing)**.

### 4. How training works (guess → check → adjust)
Training repeats this loop over every example, many times (each full pass = one *epoch*):
1. The model reads a URL and **guesses** a score (0–1).
2. We compare the guess to the **true label** — the gap is the **loss** (binary cross-entropy).
3. The optimizer (**Adam**) **adjusts** the model's internal numbers a little to reduce the loss.
4. Repeat for the next URL… for all URLs… for 15 epochs.

`EarlyStopping` keeps the best version and stops when it no longer improves. The result is a
trained "brain" saved as `phishing_url_model.keras`.

### 5. The pipeline in this notebook
| Section | What it does |
|---|---|
| 1 | Mount Google Drive (read the dataset) |
| 2 | Build the dataset: phishing URLs + real legit URLs/domains (labels 1 / 0) |
| 3 | Balance the two classes + split into train / validation / test |
| 4 | Normalize each URL + turn its characters into numbers (tokenize) |
| 5 | Build the neural network (Embedding → Conv1D → Transformer attention → output) and **train** it |
| 6 | Evaluate accuracy, precision/recall, confusion matrix on unseen test data |
| 7 | Generalization test on real-world URLs it never saw |
| 8 | Export the model + tokenizer for the backend |

### 6. How we know it actually learned (not memorized)
Section 6 measures **accuracy / precision / recall** on a **test set it never trained on**, and
Section 7 checks **brand-new real URLs**. Phishing should score high, legit should score low —
that proves it learned the real signal, not a shortcut.

## SECTION 1: Mount Google Drive

Gives the notebook access to the dataset stored in your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## SECTION 2: Build a GLOBAL dataset (phishing + worldwide legit domains)

To make the model generalize worldwide (so it does NOT need any hard-coded allowlist),
the benign side is built from three diverse sources:
1. Benign URLs from `malicious_phish.csv` (full URLs with paths).
2. **Top global domains** — auto-downloaded Cisco Umbrella top-1M (popular sites worldwide).
3. **~9,000 university domains worldwide** — auto-downloaded (covers `.edu`, `.edu.bd`,
   `.ac.uk`, etc.). This teaches the model the *pattern* of educational/government sites, so
   it correctly trusts unseen schools/universities (like `wub.edu.bd`) on its own.

Phishing = the phishing rows from the CSV. All three benign sources are mixed in balanced
proportions so educational/global domains are well represented during training.

In [ ]:
import pandas as pd
import io, json, zipfile, urllib.request

CSV_PATH = '/content/drive/MyDrive/malicious_phish.csv'   # change if needed
TOP_DOMAINS_N = 40000      # popular global domains to include
CSV_BENIGN_CAP = 40000     # cap generic benign so edu/global stay well represented


def _fetch(url, timeout=90):
    return urllib.request.urlopen(url, timeout=timeout).read()


# ---- 1. Phishing dataset (benign / phishing URLs) ----
raw = pd.read_csv(CSV_PATH)
raw.columns = [c.strip().lower() for c in raw.columns]
url_col = 'url' if 'url' in raw.columns else raw.columns[0]
label_col = next((c for c in ['type', 'label', 'result', 'class'] if c in raw.columns),
                 raw.columns[-1])


def _is_benign(v):
    return str(v).strip().lower() in ('benign', 'legitimate', 'legit', 'good', 'safe', '0')


def _is_phish(v):
    return str(v).strip().lower() in ('phishing', 'phish', 'malicious', 'bad', '1')


csv_benign = raw.loc[raw[label_col].map(_is_benign), url_col].astype(str)
csv_phish = raw.loc[raw[label_col].map(_is_phish), url_col].astype(str)
print(f'From CSV: {len(csv_benign)} benign, {len(csv_phish)} phishing')

# ---- 2. Top global domains (popular sites worldwide) ----
top_domains = []
try:
    blob = _fetch('http://s3-us-west-1.amazonaws.com/umbrella-static/top-1m.csv.zip')
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        with z.open(z.namelist()[0]) as fh:
            top = pd.read_csv(fh, header=None, names=['rank', 'domain'])
    top_domains = ['https://' + d for d in top['domain'].head(TOP_DOMAINS_N).tolist()]
    print(f'Downloaded {len(top_domains)} top global domains.')
except Exception as e:
    print('Top-domains download failed:', repr(e)[:120])

# ---- 3. Worldwide UNIVERSITY domains (teaches the edu/gov pattern globally) ----
uni_domains = []
try:
    src = ('https://raw.githubusercontent.com/Hipo/university-domains-list/master/'
           'world_universities_and_domains.json')
    unis = json.loads(_fetch(src).decode('utf-8'))
    for rec in unis:
        for d in rec.get('domains', []):
            if d:
                uni_domains.append('https://' + d.strip().lower())
    uni_domains = sorted(set(uni_domains))
    print(f'Downloaded {len(uni_domains)} university domains worldwide.')
except Exception as e:
    print('University-list download failed:', repr(e)[:120])

# ---- 4. Combine into a balanced benign pool ----
benign_pool = pd.concat([
    pd.Series(uni_domains),                                              # all universities
    pd.Series(top_domains),                                             # all top global
    csv_benign.sample(min(CSV_BENIGN_CAP, len(csv_benign)), random_state=42),  # capped generic
], ignore_index=True).drop_duplicates()

phish_urls = csv_phish.drop_duplicates()

data = pd.concat([
    pd.DataFrame({'url': benign_pool, 'y': 0}),
    pd.DataFrame({'url': phish_urls, 'y': 1}),
], ignore_index=True)
data = data[data['url'].str.len() > 3]

print(f"\nBenign pool: {int((data.y == 0).sum())} "
      f"(universities + global + generic) | phishing: {int((data.y == 1).sum())}")
print('\nSample benign (should include universities & global sites):')
print(data[data.y == 0].sample(min(6, int((data.y == 0).sum())), random_state=3)['url'].to_string(index=False))
print('Sample phishing:')
print(data[data.y == 1].sample(min(4, int((data.y == 1).sum())), random_state=3)['url'].to_string(index=False))

## SECTION 3: Balance the classes and split

Take an equal number of benign and phishing URLs (so the model can't cheat on class
size), then split 70 / 15 / 15 into train / validation / test.

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

SAMPLES_PER_CLASS = 40000   # raise for accuracy / lower for speed (use a GPU runtime!)

benign = data[data.y == 0]
phish = data[data.y == 1]
n = min(SAMPLES_PER_CLASS, len(benign), len(phish))
print(f'Using {n} per class ({2 * n} total).')

balanced = pd.concat([
    benign.sample(n, random_state=42),
    phish.sample(n, random_state=42),
]).sample(frac=1, random_state=42)  # shuffle

urls = balanced['url'].astype(str).tolist()
y_all = balanced['y'].to_numpy()

urls_train, urls_temp, y_train, y_temp = train_test_split(
    urls, y_all, test_size=0.3, random_state=42, stratify=y_all)
urls_val, urls_test, y_val, y_test = train_test_split(
    urls_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
print(f'Train {len(urls_train)} | Val {len(urls_val)} | Test {len(urls_test)}')

## SECTION 4: Normalize URLs + character tokenizer

`normalize_url` strips formatting artifacts (scheme, `www.`, case) so the model learns real
content. **This is identical to the backend's `ml_service.py`** so training and the app match.
Then each URL becomes a fixed-length sequence of character ids (id 0 = pad, id 1 = unknown).

In [ ]:
import numpy as np

MAXLEN = 200


def normalize_url(url):
    u = (url or '').strip().lower()
    if '://' in u:
        u = u.split('://', 1)[1]   # drop http:// / https://
    while u.startswith('www.'):
        u = u[4:]                  # drop leading www.
    return u


# Build vocab from TRAIN urls only (no leakage). chars start at id 2.
chars = sorted(set(''.join(normalize_url(u) for u in urls_train)))
char_index = {c: i + 2 for i, c in enumerate(chars)}
VOCAB_SIZE = len(char_index) + 2
print(f'Vocab size = {VOCAB_SIZE}, maxlen = {MAXLEN}')


def encode(url):
    seq = [char_index.get(ch, 1) for ch in normalize_url(url)[:MAXLEN]]
    return seq + [0] * (MAXLEN - len(seq))


X_train = np.array([encode(u) for u in urls_train], dtype=np.int32)
X_val = np.array([encode(u) for u in urls_val], dtype=np.int32)
X_test = np.array([encode(u) for u in urls_test], dtype=np.int32)
print('Encoded shapes:', X_train.shape, X_val.shape, X_test.shape)

## SECTION 5: Build and train the model (modern Transformer-style)

A character-level network with a **Transformer self-attention** block — the modern standard for
URL/text classification. `Embedding -> Conv1D (local context) -> Multi-Head Self-Attention ->
GlobalMaxPooling -> Dense`. It uses only standard Keras layers, so the backend can load it with
no custom code. Self-attention lets the model weigh suspicious parts of the URL (host, TLD, path
tokens) against each other instead of reading it left-to-right only.

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Modern attention-based (Transformer encoder) URL classifier.
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128

inputs = layers.Input(shape=(MAXLEN,))
x = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM)(inputs)
x = layers.Conv1D(EMBED_DIM, 5, padding='same', activation='relu')(x)  # adds positional context

# --- Transformer encoder block ---
attn = layers.MultiHeadAttention(num_heads=NUM_HEADS, key_dim=EMBED_DIM)(x, x)
attn = layers.Dropout(0.1)(attn)
x = layers.LayerNormalization(epsilon=1e-6)(x + attn)
ff = layers.Dense(FF_DIM, activation='relu')(x)
ff = layers.Dense(EMBED_DIM)(ff)
ff = layers.Dropout(0.1)(ff)
x = layers.LayerNormalization(epsilon=1e-6)(x + ff)
# ---------------------------------

x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs, outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-5)

print('Training...')
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr],
)
print('Done.')

## SECTION 6: Evaluate on the test set

Measures the trained model on data it never saw: accuracy, a precision/recall report, and a confusion matrix (how many phishing/safe were classified correctly vs. wrongly).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {acc:.4f}  |  Test loss: {loss:.4f}')

y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int)
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['benign', 'phishing']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['benign', 'phishing'], yticklabels=['benign', 'phishing'])
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
plt.show()

## SECTION 7: Generalization test (real-world URLs)

Tests the model on URLs NOT in the dataset — legit sites (with/without `www.`, with paths) and
clear phishing. A good model scores legit LOW and phishing HIGH regardless of `www.`.

In [ ]:
def score(u):
    return float(model.predict(np.array([encode(u)], dtype=np.int32), verbose=0)[0][0])

legit = ['https://www.google.com', 'https://google.com', 'https://github.com/login',
         'https://en.wikipedia.org/wiki/QR_code', 'https://www.amazon.com/gp/cart',
         'https://www.linkedin.com/feed', 'https://youtube.com/watch?v=abc']
phish = ['http://192.168.0.5@paypal-secure.tk/login/verify', 'http://free-gift-card.tk/claim/password',
         'http://45.137.21.9/secure/signin/confirm', 'http://bit.ly/3xPhish',
         'https://www.appleid-verify.tk/login']

print('LEGIT (want LOW):')
okl = 0
for u in legit:
    p = score(u); okl += p < 0.5; print(f'  {p*100:6.1f}%  {u}')
print('PHISHING (want HIGH):')
okp = 0
for u in phish:
    p = score(u); okp += p > 0.5; print(f'  {p*100:6.1f}%  {u}')
print(f'\nGeneralization: legit {okl}/{len(legit)}, phishing {okp}/{len(phish)}')

## SECTION 8: Export model + tokenizer (download these 2 files)

Saves `phishing_url_model.keras` and `url_tokenizer.json` to Drive, then verifies the model
separates the classes. Download BOTH into the backend folder
`qr-code-fishing-backend/app/models/ml/`.

In [ ]:
import os, json
import tensorflow as tf

out_dir = '/content/drive/MyDrive'
model_path = os.path.join(out_dir, 'phishing_url_model.keras')
tok_path = os.path.join(out_dir, 'url_tokenizer.json')

model.save(model_path)
with open(tok_path, 'w', encoding='utf-8') as f:
    json.dump({'char_index': char_index, 'maxlen': MAXLEN}, f)
print('Saved model     ->', model_path)
print('Saved tokenizer ->', tok_path)

m2 = tf.keras.models.load_model(model_path)
ben = X_test[y_test == 0][:1000]
mal = X_test[y_test == 1][:1000]
bs = float(m2.predict(ben, verbose=0).mean())
ms = float(m2.predict(mal, verbose=0).mean())
spread = ms - bs
print(f'Mean P(phishing): benign={bs:.3f}  phishing={ms:.3f}  spread={spread:.3f}')
assert spread > 0.30, f'NOT discriminating (spread={spread:.3f}); train longer / more data.'
print('OK. Download BOTH files into qr-code-fishing-backend/app/models/ml/')